### Strucutred Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LancgChain supports multiple schema types and methods for enforcing structured output.

### Pydantic 

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The Director of the movie")
    ratings:float=Field(description="The movies' rating out of 10")
    

In [ ]:
model_with_structure=model.with_structured_output(Movie)

model.invoke("Provide details about the movie Inception")

model_with_structure.invoke("Provide details about the movie Inception")

### Message output alongside parsed structure



In [ ]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    """ A movie with details. """
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description="This year the movie was released")
    director:str=Field(..., description="The Director of the movie")
    ratings:float=Field(..., description="The movies' rating out of 10")
    
model_with_structure = model.with_structured_output(Movie, include_raw=True)

response =model_with_structure.invoke("Provide details about the movie Inception")
response

### Nested Structure


In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions (USD)")


model_with_structure = model.with_structured_output(MovieDetails)

response =model_with_structure.invoke("Provide details about the movie Inception")
response

## TypeDict

TypeDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [ ]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """ A movie with details. """
    title:Annotated[str, ..., "The title of the movie"]
    year:Annotated[int, ..., "This year the movie was released"]
    director:Annotated[str, ..., "The Director of the movie"]
    ratings:Annotated[str, ..., "The movies' rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
response= model_withtypedict.invoke("Please provide the details of teh movie avengers")
response

In [ ]:
from typing_extensions import TypedDict, Annotated

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions (USD)")


model_with_structure = model.with_structured_output(MovieDetails)

response =model_with_structure.invoke("Provide details about the movie Inception")
response

### DataClasses

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the *@dataclass*

In [1]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

#### Pydantic


In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """ Contact information of a person. """
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result) 
# AI message, Human Message, Structured response
print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone= (555) 123-4567)

name='John Doe' email='john@example.com' phone='(555) 123-4567'


#### TypeDict


In [ ]:
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """ Contact information of a person. """
    name: Annotated[str, ..., "The name of the person"]
    email: Annotated[str, ..., "The email address of the person"]
    phone: Annotated[str, ..., "The phone number of the person"]

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result) 
# AI message, Human Message, Structured response
print(result["structured_response"])

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='917553ce-68aa-4eb8-8dce-42accddf050b'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". The tools provided include a ContactInfo function that requires name, email, and phone. \n\nFirst, I need to parse the input. The name is John Doe. The email is clearly john@example.com. The phone number is (555) 123-4567. I should check if the phone number is in the correct format. The function parameters require all three fields, so I need to make sure none are missing.\n\nWait, the phone number has parentheses and a space. Should I format it differently? The example might just need the number as provided. The user didn\'t specify formatting changes, so I\'ll leave it as is. \n\nNow, I\'ll map each

#### Dataclass

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ Contact information for a person """
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model = "groq:qwen/qwen3-32b",
    response_format = ContactInfo # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result) 
# AI message, Human Message, Structured response
print(result["structured_response"])